# Negotiate, Then Solve: training the request reader (4B, two T4s)

Trains one LoRA adapter of Qwen3.5-4B on the uploaded rows of its `TASKS` (by default `compile`: reading a
request into intents, the rows of `training.dataset`'s recipe, exported with `training.distill export
--readable` into `data/multitask-readable`). This run: **nts-intents-v8**, from the base on all of
`data/recipes/intents-v8.toml` (v7's rows, every one kept, and the kinds taught mostly by messages of several
requests topped up: `training.balance`). Not an update: v7, an update of v6 on 335 new rows and 670 replayed,
took the new rows' habits for the reading (every move a wish, cancellations without their time).

It trains the way the agent notebook (agentic branch, `train_agent_kaggle.ipynb`) found fastest on two T4s:

* **Both GPUs compute** (`PARALLEL = "data"`): in fp16 the model (~8.5 GB) fits on one T4, so training runs as
  `/tmp/nts_train.py` under `torchrun`, a copy of the model per GPU, each reading other rows of the stream,
  gradients averaged. The agent run took 68 min this way against 179 min with the model split across the
  GPUs (only one computing at a time). If it fails (out of memory), the notebook trains split instead, in the
  same session.
* **Logits on the learned positions only** (`logits_to_keep`): the 248k-word output layer is computed for the
  JSON answer's tokens, not the prompt's.
* **flash-linear-attention** (`FAST_KERNELS`) for Qwen3.5's linear-attention layers instead of transformers'
  PyTorch fallback, and the full-attention layers in blocks (`chunked_attention`, equal to SDPA to 1e-7; it
  matters for long rows, not for these).
* **fp16 from a lower loss scale** (`INIT_SCALE = 2^8`): the default 65536 overflowed and skipped the first
  steps; the run stops if no step updates the adapter for 10 steps.

The rows are a prompt (rendered with the chat template, thinking off, as `llama-server --jinja` renders a
request) and the JSON answer; the loss is on the answer and the end token only, as TRL's
`completion_only_loss` did in earlier versions of this notebook.

**Before running**

1. Zip `data/multitask-readable/` (`data/multitask-readable-intents-v8.zip`) and upload it as a Kaggle
   dataset with *multitask* in its name; add it (*Add Input*), and remove older versions of it.
2. `INIT_FROM = None`: no earlier output is needed (with `"auto"`, add the earlier run's output as an input;
   the data cell stops if far more rows are new than `EXPECT_NEW`).
3. *Settings*: **Accelerator: GPU T4 x2**, **Internet: on**.
4. *Save Version* > *Save & Run All (Commit)*. `TRIAL = True` runs 20 steps and prints seconds per step and
   peak memory, if a new setting needs checking first.
5. Serve the GGUF beside the other adapters (below) and score it:
   `python -m evaluation.suites --parser local --local <url> --adapter nts-intents-v8-read-100`.

**Learning curve.** The training stream is ordered so that every prefix of it has the same mix of tasks as
the whole, and the adapter is saved after each fraction in `CURVE` of what it reads; with a curve the learning
rate is constant after warm-up, so the points differ only in how much was read. Score each point's GGUF.

**Updates.** `INIT_FROM` starts from an earlier adapter and reads only the rows it has not seen
(`trained.json`), with `REPLAY` times as many it has, so it does not forget them. What it reads is mostly the
new rows: when they are about kinds of request the old rows teach, their habits become the reading (v7), so
train from the base then, and update only for what is new in kind.


In [ ]:
%%capture
FAST_KERNELS = True   # also install flash-linear-attention: Triton kernels for the linear-attention layers (they run on a T4)
# Keep Kaggle's CUDA build of torch: a dependency must not swap it for another build.
!pip freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/torch-pin.txt
!pip install --upgrade -c /tmp/torch-pin.txt "transformers>=5" peft trl accelerate datasets
# Kaggle ships torchao 0.10; PEFT refuses torchao < 0.16 when it adds LoRA layers. Nothing here uses it.
!pip uninstall -y -q torchao
if FAST_KERNELS:
    !pip install -c /tmp/torch-pin.txt flash-linear-attention

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch, transformers, peft, trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, (
    "Qwen3.5 needs transformers v5. Restart the kernel (Run > Restart) and run from this cell.")
N_GPU = torch.cuda.device_count()
assert N_GPU > 0, (
    f"no GPU (torch {torch.__version__}, CUDA {torch.version.cuda}): this session runs on Kaggle's CPU image. "
    "Set Settings > Accelerator > GPU T4 x2 for this notebook (and check it again in Save Version). "
    "If it is set, another GPU session (e.g. the 2B run) may be using your GPU: wait for it to finish.")
for i in range(N_GPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")
if N_GPU < 2:
    print("WARNING: one GPU. Set Accelerator: GPU T4 x2; the 4B model in float32 does not fit on one T4.")

## Settings

In [ ]:
import glob, json, math, re, time
from pathlib import Path

SESSION_START = time.time()  # Kaggle's 12 hours count from here, not from training

MODEL = "Qwen/Qwen3.5-4B"
# What to train: one LoRA adapter on some tasks of the uploaded rows (adapters.toml).
ADAPTER = "nts-intents-v8"  # its name: the GGUF file and the name in adapters.toml
TASKS = ["compile"]         # the rows' tasks it learns: compile, policy, reply, explain (None: all of them)
# Update an earlier version instead of training from the base: its output folder (attach that notebook output or
# a dataset: the folder with trained.json, and the adapter in lora/ or beside it). It reads the rows that version
# has not seen and REPLAY times as many that it has. None: a new adapter. "auto": the one attached input that has
# a trained.json. This run: from the base on all of v8's data (data/recipes/intents-v8.toml).
INIT_FROM = None
REPLAY = 2.0                # twice the new rows again from the ones it has seen, so it keeps them
EXPECT_NEW = None           # with INIT_FROM: about how many rows are new (intents-v7 added 335 to v6's 2719)
MERGE = False               # the server applies adapters per request (--lora ... --lora-init-without-apply)
TRIAL = False               # True: 20 steps, time projection
PRECISION = "fp16"          # "fp16" (T4 tensor cores; the v6 run trained in it) or "fp32" (model split only)
EPOCHS = 3
LR = 2e-4
MAX_LEN = 512               # schema 4 rows: the longest compile row is under 300 tokens
BATCH, ACCUM = 4, 4         # 16 rows per optimizer step on any number of GPUs (ACCUM is divided among them)
WARMUP = 5                  # optimizer steps
PARALLEL = "data"           # "data": a copy of the model per GPU, both compute (fp16 only); "model": split across them
SEED = 3407
INIT_SCALE = 2.0 ** 8       # fp16 loss scale at the start (the default 65536 overflows and skips steps)
# Weights per GPU when the model is split (PARALLEL = "model", or the fallback). The output layer is tied to the
# embeddings, so it sits on GPU 0: keep GPU 0 lighter.
MAX_MEMORY = {0: "8GiB", 1: "12GiB"}
TIME_BUDGET_H = 10.75       # hours since the session started: stop with a checkpoint before Kaggle's 12
RESUME = True               # continue from the latest checkpoint in an attached earlier output
TRAIN_EXAMPLES = None       # train on this many examples (each task in proportion); None: all of them
READ = 1.0
CURVE = [1/3, 2/3, 1.0]     # an adapter after each epoch: score all three, later runs use the fewest that do as well
CURVE_QUANTS = ["Q8_0"]
R_DPO = None                # the reader has no preference pairs (the agent notebook's DPO stage is not here)

found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
assert found, "no train.jsonl under /kaggle/input: add the multitask dataset (Add Input)"
multitask = [f for f in found if "multitask" in f.lower()]
if not multitask and len(found) > 1:
    print("WARNING: no dataset path contains 'multitask'; using", found[0], "of", found)
DATA = Path((multitask or found)[0]).parent
NAME = ADAPTER
OUT = Path("/kaggle/working") / NAME
if INIT_FROM == "auto":
    earlier = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/trained.json", recursive=True)})
    assert len(earlier) == 1, (f"INIT_FROM = 'auto' needs exactly one attached output with a trained.json, found "
                               f"{earlier or 'none'}: attach the earlier version's output, or name its folder")
    INIT_FROM = earlier[0]
if INIT_FROM:
    INIT_FROM = Path(INIT_FROM)
    if not (INIT_FROM / "trained.json").exists():  # a notebook output's or an input's root: find the adapter in it
        INIT_FROM = Path(next(glob.iglob(f"{INIT_FROM}/**/trained.json", recursive=True))).parent
    INIT_LORA = INIT_FROM / "lora" if (INIT_FROM / "lora").exists() else INIT_FROM
    print("updating", json.loads((INIT_FROM / "manifest.json").read_text())["adapter"], "from", INIT_FROM)
print("data:", DATA, "| output:", OUT)


In [ ]:
%%writefile /tmp/nts_train.py
"""The training run of the 4B notebooks (the request reader's and the agent's), written to /tmp/nts_train.py.

    torchrun --nproc_per_node 2 /tmp/nts_train.py run.json    # data parallel: a copy of the model per GPU
    nts_train.train(C, load_model(C, "auto", ...), ...)      # in the notebook: the model split across the GPUs

Data parallel needs the model to fit on one GPU (fp16: ~8.5 GB on a 15 GB T4) and keeps both GPUs computing:
each reads other episodes of the stream, in its order, and the gradients are averaged. Split, only one GPU
computes at a time. ``run.json`` holds the notebook's settings; the stream and the validation episodes are
on disk (``datasets.save_to_disk``); the result (steps, losses, seconds per step) goes to ``result.json``.
With ``"MODE": "dpo"`` it trains the SFT adapter further on preference pairs (``train_dpo``) and writes
``lora-dpo/`` and ``result-dpo.json``.
"""
import json
import math
import os
import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_from_disk
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import SequentialSampler
from transformers import AutoModelForImageTextToText, Trainer, TrainerCallback, TrainingArguments

TARGETS = r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"
CHUNK = 1024  # queries per block of the chunked attention
# fp16 loss scale at the start. The default (65536) overflows on these episodes and halves once per skipped step:
# the first run skipped 7 steps and settled at 512 (halving once more later). From a halving below that, the
# first steps already update the adapter.
INIT_SCALE = 2.0 ** 8


def chunked_attention(module, query, key, value, attention_mask, scaling=None, dropout=0.0, **kwargs):
    """Causal attention a block of queries at a time, each block recomputed in the backward pass.

    On a T4, SDPA has no fused kernel that trains with Qwen3.5's head_dim 256 (flash attention needs sm80), so it
    falls back to the math kernel and keeps the whole score matrix: heads x n x n in fp32, 1.5 GB per layer
    for a 5k-token episode, which ran one GPU out of memory. Here a block holds CHUNK x n scores at a time, and
    only the keys its queries can see (half the work of the full matrix). query (B, H, L, D), key and value
    (B, H_kv, S, D); with a KV cache (generation) the queries are the last L of S positions."""
    from torch.utils.checkpoint import checkpoint

    rep = query.shape[1] // key.shape[1]
    if rep > 1:
        key, value = key.repeat_interleave(rep, 1), value.repeat_interleave(rep, 1)
    L, S = query.shape[2], key.shape[2]
    scale = scaling if scaling is not None else query.shape[-1] ** -0.5
    pos_k = torch.arange(S, device=query.device)

    def block(q, k, v, allowed):
        s = torch.matmul(q, k.transpose(-1, -2)).float() * scale
        s = s.masked_fill(~allowed, float("-inf"))
        return torch.matmul(torch.softmax(s, -1).to(v.dtype), v)

    out = []
    for i in range(0, L, CHUNK):
        j = min(i + CHUNK, L)
        last = S - L + j  # keys beyond the block's last query are never visible
        allowed = pos_k[None, :last] <= (S - L + torch.arange(i, j, device=query.device))[:, None]
        if attention_mask is not None:  # a 4D mask from transformers: bool (True = attend) or additive
            m = attention_mask[:, :, i:j, :last]
            allowed = allowed & (m if m.dtype == torch.bool else m == 0)
        args = (query[:, :, i:j], key[:, :, :last], value[:, :, :last], allowed)
        out.append(checkpoint(block, *args, use_reentrant=False) if torch.is_grad_enabled() else block(*args))
    return torch.cat(out, 2).transpose(1, 2).contiguous(), None


def register_attention():
    from transformers import AttentionInterface
    from transformers.masking_utils import AttentionMaskInterface, sdpa_mask
    AttentionInterface.register("nts_chunked", chunked_attention)
    AttentionMaskInterface.register("nts_chunked", sdpa_mask)


def load_model(C, device_map, max_memory=None, adapter=None, trainable=True):
    """The base model with LoRA on the language layers: a new adapter, or ``adapter`` (a saved one)."""
    dtype = torch.float32 if C["PRECISION"] == "fp32" else torch.float16
    register_attention()
    model = AutoModelForImageTextToText.from_pretrained(C["MODEL"], dtype=dtype, device_map=device_map,
                                                        max_memory=max_memory, attn_implementation="nts_chunked")
    model.config.use_cache = False
    if adapter:
        return PeftModel.from_pretrained(model, str(adapter), is_trainable=trainable)
    return get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
                                            task_type="CAUSAL_LM", target_modules=TARGETS))


def collator(pad):
    def collate(batch):
        """Right-padded input_ids, attention_mask and labels (-100 on padding)."""
        n = max(len(x["input_ids"]) for x in batch)
        ids = torch.tensor([x["input_ids"] + [pad] * (n - len(x["input_ids"])) for x in batch])
        labels = torch.tensor([x["labels"] + [-100] * (n - len(x["labels"])) for x in batch])
        mask = torch.tensor([[1] * len(x["input_ids"]) + [0] * (n - len(x["input_ids"])) for x in batch])
        return {"input_ids": ids, "attention_mask": mask, "labels": labels}
    return collate


class Episodes(Trainer):
    """The loss on the learned tokens only, with logits for those positions only: the output layer (248k
    vocabulary) over a whole episode would take GBs in fp32, and about 5% of the tokens are learned. The loss
    is the mean over each episode's learned tokens."""
    full_logits = False  # set if the model does not take positions in logits_to_keep

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        target = labels[:, 1:]                                    # position p predicts token p + 1
        # a list, not a tensor: accelerate's hooks move tensor arguments to GPU 0, but the output layer may sit on
        # another GPU, and a tensor index must be on the indexed tensor's device
        keep = (target != -100).any(0).nonzero().squeeze(1).tolist()
        if not Episodes.full_logits:
            try:
                out = model(**inputs, logits_to_keep=keep, use_cache=False)
                logits, target = out.logits, target[:, keep]
            except TypeError:
                Episodes.full_logits = True
                print("WARNING: the model takes no positions in logits_to_keep: full logits (more memory)")
        if Episodes.full_logits:
            out = model(**inputs, use_cache=False)
            logits = out.logits[:, :-1]
        target = target.to(logits.device)
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), target.reshape(-1), ignore_index=-100)
        return (loss, out) if return_outputs else loss


class InOrder(Episodes):
    """Reads the stream in the order built in the data cell (a Trainer shuffles by default), so the adapter
    saved after a fraction of it has seen that fraction of every task. With several GPUs, accelerate deals
    the batches out in turn, so together they still read the stream in order."""
    def _get_train_sampler(self, *args, **kwargs):
        return SequentialSampler(self.train_dataset)


class Curve(TrainerCallback):
    """Saves the adapter at each curve point, apart from the rotating checkpoints."""
    def __init__(self, C, tok):
        self.C, self.tok, self.steps = C, tok, {int(k): v for k, v in C["CURVE_STEPS"].items()}

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step in self.steps and state.is_world_process_zero:
            f = self.steps[state.global_step]
            path = Path(self.C["OUT"]) / "curve" / f"read-{round(f * 100):03d}"
            model.save_pretrained(str(path))
            self.tok.save_pretrained(str(path))
            (path / "read.json").write_text(json.dumps({
                "fraction": f, "step": state.global_step, "examples": state.global_step * self.C["PER_STEP"],
                "epochs": self.C["EPOCHS"], "read": self.C["READ"], "loss": next(
                    (h["loss"] for h in reversed(state.log_history) if "loss" in h), None)}))
            print(f"curve: adapter after {f:.0%} of the reading saved to {path}")


class Clock(TrainerCallback):
    """Seconds per optimizer step, a stop with a checkpoint before the session limit, a stop if the loss is
    not finite or no step updates the adapter (fp16 overflow), and peak memory for the first steps."""
    def __init__(self, C):
        self.C, self.marks, self.skipped = C, [], 0

    def on_step_end(self, args, state, control, **kwargs):
        self.marks.append(time.time())
        if not self.C["TRIAL"] and (time.time() - self.C["SESSION_START"]) / 3600 > self.C["TIME_BUDGET_H"]:
            print(f"time budget reached at step {state.global_step}: saving a checkpoint and stopping")
            control.should_save = control.should_training_stop = True

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and not math.isfinite(logs["loss"]):
            print("loss is not finite (fp16 overflows here): use PRECISION = 'fp32'")
            control.should_training_stop = True
        if logs and "grad_norm" in logs:  # a skipped step (fp16 overflow) logs a nan gradient norm
            self.skipped = 0 if math.isfinite(logs["grad_norm"]) else self.skipped + 1
            if self.skipped * args.logging_steps >= 10:
                print(f"no update in the last {self.skipped * args.logging_steps} steps (the fp16 loss scale keeps "
                      "overflowing): lower INIT_SCALE or use PRECISION = 'fp32'")
                control.should_training_stop = True
        if (logs and "loss" in logs and state.global_step <= 10 and torch.cuda.is_available()
                and state.is_world_process_zero):
            print(f"step {state.global_step}: loss {logs['loss']:.3f}, " + ", ".join(
                f"GPU {i} peak {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GB" for i in range(torch.cuda.device_count())))

    def per_step(self):
        m = self.marks[5:] if len(self.marks) > 8 else self.marks  # skip warm-up steps
        return (m[-1] - m[0]) / max(len(m) - 1, 1)


def train(C, model, tok, stream, val, processes=1):
    """Trains ``model`` on ``stream``; returns the result the notebook reads (also in OUT/result.json)."""
    model.enable_input_require_grads()  # gradient checkpointing with frozen embeddings
    clock = Clock(C)
    curve = bool(C["CURVE_STEPS"])
    args = dict(
        output_dir = str(Path(C["OUT"]) / "checkpoints"),
        remove_unused_columns = False,        # the collator reads input_ids and labels itself
        prediction_loss_only = True,          # evaluation: the loss only, no logits gathered
        per_device_train_batch_size = C["BATCH"],
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = C["ACCUM"] // processes,  # the same episodes per step on any number of GPUs
        num_train_epochs = 1 if curve else C["EPOCHS"],  # the curve stream already holds the epochs
        max_steps = 20 if C["TRIAL"] else -1,
        learning_rate = C["LR"],
        lr_scheduler_type = "constant_with_warmup" if curve else "cosine",  # comparable curve points
        warmup_steps = C["WARMUP"],
        weight_decay = 0.01,
        optim = "adamw_torch",                # only the adapters train: the optimizer state is small
        fp16 = C["PRECISION"] == "fp16",
        bf16 = False,                         # T4 has no bf16
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        ddp_find_unused_parameters = False,   # every LoRA layer is used (the vision tower has none)
        logging_steps = 2,
        eval_strategy = "no" if C["TRIAL"] else "steps",
        eval_steps = C["STEPS_PER_EPOCH"],
        save_strategy = "no" if C["TRIAL"] else "steps",
        save_steps = 25,
        save_total_limit = 2,
        report_to = "none",
        seed = C["SEED"],
    )
    args.update(C.get("ARGS") or {})
    trainer = (InOrder if curve else Episodes)(
        model=model, processing_class=tok, train_dataset=stream, eval_dataset=val,
        data_collator=collator(C["PAD"]), callbacks=[clock, Curve(C, tok)], args=TrainingArguments(**args))
    trainer.model_accepts_loss_kwargs = False  # compute_loss returns a mean: the Trainer divides by the accumulation
    if trainer.accelerator.scaler is not None:  # fp16; read when train() prepares the optimizer, a resume loads its state
        trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=C.get("INIT_SCALE", INIT_SCALE))
    main = trainer.is_world_process_zero()
    if main:
        print(f"{processes} process(es); model parallel across GPUs: {trainer.is_model_parallel}; "
              f"{C['BATCH']} x {args['gradient_accumulation_steps']} x {processes} episodes per step")
    if curve:  # the trainer must read the stream in order: the first batch is the stream's first example
        first = next(iter(trainer.get_train_dataloader()))["input_ids"][0].tolist()
        want = trainer.train_dataset[0]["input_ids"]
        assert not main or first[: len(want)] == want, "the trainer shuffles the stream: the curve points would not be prefixes"
    stats = trainer.train(resume_from_checkpoint=C.get("RESUME_FROM"))
    done = trainer.state.global_step >= trainer.state.max_steps
    result = {
        "processes": processes, "steps": trainer.state.global_step, "max_steps": trainer.state.max_steps,
        "done": done, "minutes": stats.metrics["train_runtime"] / 60, "per_step": clock.per_step(),
        "losses": [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h],
        "validation": [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h],
        "peak_gb": [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())]}
    if done or C["TRIAL"]:  # the adapter the quick check and the export read
        trainer.save_model(str(Path(C["OUT"]) / "lora"))
    if main:
        (Path(C["OUT"]) / "result.json").write_text(json.dumps(result))
    return result


def pair_collator(pad):
    def collate(batch):
        """A pair as one batch of two sequences, chosen then rejected (right-padded): one forward for both, so
        data parallel sees each parameter used once per step. One pair per micro-batch."""
        assert len(batch) == 1, "DPO reads one pair per micro-batch"
        x = batch[0]
        seqs = [(x["chosen_ids"], x["chosen_labels"]), (x["rejected_ids"], x["rejected_labels"])]
        n = max(len(i) for i, _ in seqs)
        return {"input_ids": torch.tensor([i + [pad] * (n - len(i)) for i, _ in seqs]),
                "attention_mask": torch.tensor([[1] * len(i) + [0] * (n - len(i)) for i, _ in seqs]),
                "labels": torch.tensor([lab + [-100] * (n - len(lab)) for _, lab in seqs]),
                "ref": torch.tensor([x["ref_chosen"], x["ref_rejected"]], dtype=torch.float32)}
    return collate


def seq_logp(model, input_ids, attention_mask, labels):
    """Each row's summed log-probability of its learned tokens (one assistant turn), from the logits at those
    positions only."""
    target = labels[:, 1:]
    keep = (target != -100).any(0).nonzero().squeeze(1).tolist()
    out = model(input_ids=input_ids, attention_mask=attention_mask, logits_to_keep=keep, use_cache=False)
    t = target[:, keep].to(out.logits.device)
    logp = torch.log_softmax(out.logits.float(), -1).gather(-1, t.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    return (logp * (t != -100)).sum(-1)


class Preferences(Trainer):
    """DPO: -log sigmoid(beta * ((chosen - ref chosen) - (rejected - ref rejected))), with the reference's
    log-probabilities computed before training (the SFT adapter) and carried with each pair."""
    beta = 0.1
    seen = []  # (loss, margin) per pair, on this process

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        ref = inputs.pop("ref")
        logp = seq_logp(model, inputs["input_ids"], inputs["attention_mask"], inputs["labels"])
        ref = ref.to(logp.device)
        margin = (logp[0] - ref[0]) - (logp[1] - ref[1])
        loss = -F.logsigmoid(self.beta * margin)
        Preferences.seen.append((loss.item(), margin.item()))
        return loss


def pair_accuracy(model, pairs, pad):
    """The share of pairs whose chosen side the model prefers more than the reference did (margin > 0)."""
    collate, wins = pair_collator(pad), 0
    model.eval()
    with torch.no_grad():
        for x in pairs:
            b = collate([x])
            dev = model.get_input_embeddings().weight.device
            logp = seq_logp(model, b["input_ids"].to(dev), b["attention_mask"].to(dev), b["labels"].to(dev))
            wins += float((logp[0] - b["ref"][0].to(logp.device)) - (logp[1] - b["ref"][1].to(logp.device))) > 0
    return wins / max(len(pairs), 1)


def train_dpo(C, model, tok, pairs, val, processes=1):
    """DPO on ``pairs`` (with reference log-probabilities); saves lora-dpo/ and returns result-dpo.json."""
    model.enable_input_require_grads()
    Preferences.beta, Preferences.seen = C["DPO_BETA"], []
    clock = Clock({**C, "TRIAL": False})
    args = TrainingArguments(
        output_dir=str(Path(C["OUT"]) / "checkpoints-dpo"), remove_unused_columns=False, prediction_loss_only=True,
        per_device_train_batch_size=1, gradient_accumulation_steps=max(1, C["DPO_ACCUM"] // processes),
        num_train_epochs=C["DPO_EPOCHS"], learning_rate=C["DPO_LR"], lr_scheduler_type="cosine", warmup_steps=2,
        weight_decay=0.0, optim="adamw_torch", fp16=C["PRECISION"] == "fp16", bf16=False,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        ddp_find_unused_parameters=False, logging_steps=2, eval_strategy="no", save_strategy="no",
        report_to="none", seed=C["SEED"], disable_tqdm=True)
    trainer = Preferences(model=model, processing_class=tok, train_dataset=pairs, data_collator=pair_collator(C["PAD"]),
                          callbacks=[clock], args=args)
    trainer.model_accepts_loss_kwargs = False
    if trainer.accelerator.scaler is not None:
        trainer.accelerator.scaler = torch.amp.GradScaler("cuda", init_scale=C.get("INIT_SCALE", INIT_SCALE))
    stats = trainer.train()
    main = trainer.is_world_process_zero()
    seen = Preferences.seen
    result = {"processes": processes, "steps": trainer.state.global_step, "pairs": len(pairs),
              "minutes": stats.metrics["train_runtime"] / 60, "per_step": clock.per_step(),
              "losses": [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h],
              "first_margins_positive": round(sum(m > 0 for _, m in seen[:len(seen) // 4]) / max(len(seen) // 4, 1), 3),
              "last_margins_positive": round(sum(m > 0 for _, m in seen[-(len(seen) // 4):]) / max(len(seen) // 4, 1), 3)}
    if main:
        trainer.save_model(str(Path(C["OUT"]) / "lora-dpo"))
        if len(val):
            result["val_pairs"] = len(val)
            result["val_accuracy"] = round(pair_accuracy(trainer.model, val, C["PAD"]), 3)
        (Path(C["OUT"]) / "result-dpo.json").write_text(json.dumps(result))
    return result


def main(path):
    C = json.loads(Path(path).read_text())
    rank, world = int(os.environ.get("LOCAL_RANK", 0)), int(os.environ.get("WORLD_SIZE", 1))
    if torch.cuda.is_available():
        torch.cuda.set_device(rank)
        device_map = {"": rank}
    else:
        device_map = {"": "cpu"}
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(C["MODEL"])
    if C.get("MODE") == "dpo":  # the SFT adapter, trained further
        model = load_model(C, device_map, adapter=Path(C["OUT"]) / "lora")
        train_dpo(C, model, tok, load_from_disk(C["PAIRS"]), load_from_disk(C["PAIRS_VAL"]), processes=world)
        return
    model = load_model(C, device_map, adapter=C.get("INIT_LORA"))
    if rank == 0:
        model.print_trainable_parameters()
    train(C, model, tok, load_from_disk(C["STREAM"]), load_from_disk(C["VAL"]), processes=world)


if __name__ == "__main__":
    main(sys.argv[1])


## Model and LoRA adapter

Qwen3.5 is a vision-language model (`Qwen3_5ForConditionalGeneration`). The whole
model is loaded, so the merged checkpoint keeps its architecture for the GGUF
conversion, but LoRA goes on the language layers only (the same modules as the 2B
run) and the vision tower stays frozen and unused. With `INIT_FROM` the earlier adapter is
loaded, trainable, instead of a new one. With `PARALLEL = "data"` the training processes load it, not
this cell.

In [ ]:
import sys
from transformers import AutoTokenizer

sys.path.insert(0, "/tmp")
import nts_train  # written by the cell above

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.chat_template is None:  # some releases keep the template with the processor
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODEL).chat_template
print("eos:", repr(tok.eos_token))

# data parallel: each training process loads its own copy of the model on its GPU (the Train cell); none here
DATA_PARALLEL = PARALLEL == "data" and N_GPU > 1 and PRECISION == "fp16"
model = None
if not DATA_PARALLEL:
    model = nts_train.load_model({"MODEL": MODEL, "PRECISION": PRECISION}, "auto", MAX_MEMORY,
                                   adapter=INIT_LORA if INIT_FROM else None)
    placed = {str(d) for d in model.hf_device_map.values()}
    print("devices:", sorted(placed))
    assert not placed & {"cpu", "disk"}, (
        "part of the model was offloaded to the CPU (very slow): raise MAX_MEMORY or use PRECISION = 'fp16'")
    for i in range(N_GPU):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB of weights")
    model.print_trainable_parameters()
print("training:", f"data parallel, a copy of the model on each of {N_GPU} GPUs" if DATA_PARALLEL else
      "one process, the model split across the GPUs")


## Data

Each row is its prompt, rendered with the chat template and **thinking disabled** exactly as the local server
renders it, followed by the JSON answer and the end token; the labels are the answer's tokens (-100 on the
prompt). Only the rows of `TASKS` are kept; with `INIT_FROM`, only the rows the earlier version has not seen (a
row is known by a hash of its messages, so a changed row counts as new) and `REPLAY` times as many it has.


In [ ]:
import json, hashlib, random
from collections import Counter, defaultdict
from datasets import Dataset

def render_prompt(messages):
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)

def row_hash(row):
    return hashlib.sha1(json.dumps(row["messages"], sort_keys=True).encode()).hexdigest()[:16]

def load(name):
    with open(DATA / name, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return [r for r in rows if not TASKS or r.get("task", "compile") in TASKS]

def encode(row):
    """The prompt's tokens, then the answer's and the end token; labels on the answer and the end only."""
    p = tok(render_prompt(row["messages"]), add_special_tokens=False)["input_ids"]
    c = tok(row["messages"][2]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + c, "labels": [-100] * len(p) + c, "task": row.get("task", "compile")}

rows = {"train": load("train.jsonl"), "val": load("val.jsonl")}
SEEN = set(json.loads((INIT_FROM / "trained.json").read_text())) if INIT_FROM else set()
if INIT_FROM:
    new = [r for r in rows["train"] if row_hash(r) not in SEEN]
    old = [r for r in rows["train"] if row_hash(r) in SEEN]
    assert new, f"every row was seen by {INIT_FROM}: nothing to update"
    replay = random.Random(SEED).sample(old, min(len(old), round(REPLAY * len(new))))
    print(f"update: {len(new)} new rows, {len(replay)} replayed of {len(old)} seen")
    assert EXPECT_NEW is None or len(new) <= 1.5 * EXPECT_NEW, (
        f"{len(new)} new rows, expected about {EXPECT_NEW}: the attached version read other rows than this data "
        "builds on (another recipe or system prompt). Attach the right version, or set EXPECT_NEW = None to go on")
    rows["train"] = new + replay
TRAINED = sorted(SEEN | {row_hash(r) for r in rows["train"]})   # what this adapter has read, for the next update
ds = {k: Dataset.from_list([encode(r) for r in v]) for k, v in rows.items()}
ROWS = {"train": len(ds["train"]), "new": len(new) if INIT_FROM else len(ds["train"]), "val": len(ds["val"])}
print("train tasks:", Counter(ds["train"]["task"]), "| val tasks:", Counter(ds["val"]["task"]))
ex = ds["train"][0]
print(tok.decode(ex["input_ids"])[-500:])
print("--- learned in it:", tok.decode([t for t in ex["labels"] if t != -100]))
n_tokens = [len(x) for x in ds["train"]["input_ids"]]
print("tokens per row: max", max(n_tokens), "mean", sum(n_tokens) // len(n_tokens))
assert max(n_tokens) <= MAX_LEN, "raise MAX_LEN"

def stratified_order(tasks, seed):
    """Indices in an order where every prefix has each task in proportion to the whole: the k-th of a task's n
    examples (shuffled) is placed at (k + u) / n, u random in [0, 1)."""
    rng, by_task = random.Random(seed), defaultdict(list)
    for i, t in enumerate(tasks):
        by_task[t].append(i)
    keyed = []
    for t in sorted(by_task):
        idx = by_task[t]
        rng.shuffle(idx)
        keyed += [((k + rng.random()) / len(idx), i) for k, i in enumerate(idx)]
    return [i for _, i in sorted(keyed)]

tasks = ds["train"]["task"]
ds = {k: v.remove_columns(["task"]) for k, v in ds.items()}
if TRAIN_EXAMPLES and TRAIN_EXAMPLES < len(tasks):  # a prefix of the stratified order: every task in proportion
    keep = sorted(stratified_order(tasks, SEED - 1)[:TRAIN_EXAMPLES])
    ds["train"] = ds["train"].select(keep)
    tasks = [tasks[i] for i in keep]
STEPS_PER_EPOCH = math.ceil(len(ds["train"]) / (BATCH * ACCUM))
if CURVE:
    order = [i for e in range(EPOCHS) for i in stratified_order(tasks, SEED + e)]  # a new order each epoch
    order = order[: max(BATCH * ACCUM, round(READ * len(order)))]
    train_stream = ds["train"].select(order)
    TOTAL_STEPS = math.ceil(len(order) / (BATCH * ACCUM))
    CURVE_STEPS = {max(1, round(f * TOTAL_STEPS)): f for f in CURVE}  # optimizer step -> fraction read
    print(f"reading {len(order)} rows ({READ:g} of {EPOCHS} epochs), {TOTAL_STEPS} optimizer steps;",
          "adapters saved at steps", sorted(CURVE_STEPS))
else:
    train_stream, CURVE_STEPS = ds["train"], {}
    TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS
print("optimizer steps:", STEPS_PER_EPOCH, "per epoch,", TOTAL_STEPS, "in all")


## Train

In [ ]:
import shutil, subprocess, sys

resume = None
if RESUME and not TRIAL:
    # only this adapter's own checkpoints: the attached earlier version (INIT_FROM) has its own
    states = [s for s in glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
              if f"/{ADAPTER}/checkpoints/" in s]
    if states:
        resume = str(Path(max(states, key=lambda p: int(re.search(r"checkpoint-(\d+)", p).group(1)))).parent)
print("resume from:", resume)
if resume:  # the curve points an earlier session saved belong in this version's output too
    for done in glob.glob(f"/kaggle/input/**/{ADAPTER}/curve/read-*", recursive=True):
        shutil.copytree(done, OUT / "curve" / Path(done).name, dirs_exist_ok=True)

C = dict(MODEL=MODEL, PRECISION=PRECISION, OUT=str(OUT), ADAPTER=ADAPTER, TRIAL=TRIAL, EPOCHS=EPOCHS, READ=READ,
         CURVE_STEPS={str(k): v for k, v in CURVE_STEPS.items()}, BATCH=BATCH, ACCUM=ACCUM, PER_STEP=BATCH * ACCUM,
         LR=LR, WARMUP=WARMUP, SEED=SEED, STEPS_PER_EPOCH=STEPS_PER_EPOCH, TIME_BUDGET_H=TIME_BUDGET_H,
         SESSION_START=SESSION_START, INIT_SCALE=INIT_SCALE, PAD=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
         INIT_LORA=str(INIT_LORA) if INIT_FROM else None, RESUME_FROM=resume,
         STREAM="/tmp/nts-stream", VAL="/tmp/nts-val")
R = None
if DATA_PARALLEL:
    train_stream.save_to_disk(C["STREAM"])
    ds["val"].save_to_disk(C["VAL"])
    Path("/tmp/run.json").write_text(json.dumps({**C, "ARGS": {"disable_tqdm": True}}))
    proc = subprocess.Popen([sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(N_GPU),
                             "/tmp/nts_train.py", "/tmp/run.json"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() == 0:
        R = json.loads((OUT / "result.json").read_text())
    else:
        print(f"data-parallel training failed (exit {proc.returncode}, above): training with the model split "
              "across the GPUs instead")
        shutil.rmtree(OUT / "checkpoints", ignore_errors=True)
        shutil.rmtree(OUT / "curve", ignore_errors=True)
if R is None:  # one process, the model split across the GPUs (only one computes at a time)
    if model is None:
        model = nts_train.load_model(C, "auto", MAX_MEMORY, adapter=INIT_LORA if INIT_FROM else None)
    R = nts_train.train(C, model, tok, train_stream, ds["val"])
elif R["done"] or TRIAL:  # the trained adapter, split across the GPUs, for the quick check
    model = nts_train.load_model(C, "auto", MAX_MEMORY, adapter=OUT / "lora", trainable=False)

print(f"{R['minutes']:.1f} min this session, {R['processes']} process(es); peak GB per GPU {R['peak_gb']}")
DONE = R["done"]
if not DONE:  # the quick check and export would not fit in the session; the next one does them
    print(f"stopped at step {R['steps']} of {R['max_steps']}: add this version's output "
          "as an input and commit again to resume")
print("losses:", R["losses"])
print("validation:", R["validation"])

per_step = R["per_step"]
total = TOTAL_STEPS
hours = per_step * total / 3600
print(f"{per_step:.0f} s per optimizer step -> the full run ({total} steps, {EPOCHS} epochs) takes about {hours:.1f} h"
      " plus evaluation")
if hours > TIME_BUDGET_H:
    fit = int(TIME_BUDGET_H * 3600 / per_step / STEPS_PER_EPOCH)
    if CURVE:
        print(f"or set READ = {TIME_BUDGET_H * 3600 / per_step / total * READ:.2f} to fit in one")
    print(f"more than one {TIME_BUDGET_H:.0f}-hour session: commit twice (it resumes), "
          f"or set EPOCHS = {max(fit, 1)} to fit in one" + ("" if fit else " (one epoch already needs two sessions)"))


## Quick check on the validation split

Greedy decoding on validation rows: is the output valid JSON, and is it identical to the gold answer (per task)?
A sanity check only; the real scores come from `python -m evaluation.suites --parser local` against the served
GGUF.


In [ ]:
from collections import defaultdict

model.eval()
PER_TASK = 0 if not DONE else 2 if TRIAL else 15
val_rows, taken = [], defaultdict(int)
for line in open(DATA / "val.jsonl", encoding="utf-8"):
    row = json.loads(line)
    task = row.get("task", "compile")
    if (not TASKS or task in TASKS) and taken[task] < PER_TASK:
        taken[task] += 1
        val_rows.append(row)

def norm(o):
    """Drop empty/default fields: [] vs missing, "none" vs missing mean the same."""
    if isinstance(o, dict):
        return {k: norm(v) for k, v in o.items() if v not in (None, [], "none")}
    if isinstance(o, list):
        return [norm(x) for x in o]
    return o

score = defaultdict(lambda: [0, 0, 0])  # n, valid JSON, match
misses = []
for row in val_rows:
    task = row.get("task", "compile")
    inputs = tok(render_prompt(row["messages"]), return_tensors="pt", add_special_tokens=False).to("cuda:0")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["messages"][2]["content"])
    score[task][0] += 1
    try:
        pred = json.loads(text)
        score[task][1] += 1
        hit = norm(pred) == norm(gold)
        score[task][2] += hit
        if not hit:
            misses.append((task, row["id"], text, gold))
    except json.JSONDecodeError:
        misses.append((task, row["id"], text, gold))
for task, (n, valid, match) in sorted(score.items()):
    print(f"{task:8s} n={n:4d}  valid JSON {valid / n:.0%}  match {match / n:.0%}")
for task, rid, text, gold in misses[:8]:
    print("---", task, rid)
    print("pred:", text[:300])
    print("gold:", json.dumps(gold)[:300])


## Export

1. the PEFT adapter (`lora/`, and `lora-dpo/` after DPO), `trained.json` (the rows it has read, for an update with
   `INIT_FROM`) and `manifest.json` (adapter, base, tasks, parent, rows);
2. the adapter as a **LoRA GGUF** (`gguf/<ADAPTER>.gguf`, f16, tens of MB) with llama.cpp's
   `convert_lora_to_gguf.py`, against the base's own config; with a curve, one per point;
3. with `MERGE = True` only: the adapter merged into the bf16 base and quantized to Q8_0, as
   before (a whole model per adapter, for a server that cannot load adapters).

A TRIAL exports 1 and 2 too, from its 20 steps, to check conversion and serving early.


In [ ]:
import gc

if not DONE and not TRIAL:
    print("training not finished: no export this session")
else:
    if model is not None:  # (training saved lora/ already; after DPO the model is freed)
        model.save_pretrained(str(OUT / "lora"))
    tok.save_pretrained(str(OUT / "lora"))
    (OUT / "trained.json").write_text(json.dumps(TRAINED))
    MANIFEST = {"adapter": ADAPTER, "base": MODEL, "tasks": TASKS,
                "parent": json.loads((INIT_FROM / "manifest.json").read_text())["adapter"] if INIT_FROM else None,
                "rows": ROWS, "epochs": EPOCHS, "steps": R["steps"], "processes": R["processes"], "trial": TRIAL,
                "lr": LR, "max_len": MAX_LEN, "dpo": R_DPO}
    # the data it read, as training.dataset recorded it: recipe, code version and every split's fingerprint
    if (DATA / "compile-manifest.json").exists():
        data = json.loads((DATA / "compile-manifest.json").read_text())
        MANIFEST["data"] = {k: data.get(k) for k in ("recipe", "recipe_sha", "code", "schema", "fingerprints")}
    (OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))
    print(json.dumps(MANIFEST, indent=1))
    model = None
    gc.collect()
    torch.cuda.empty_cache()

    from peft import PeftModel
    from transformers import AutoModelForImageTextToText
    from transformers import AutoProcessor

    def merge(adapter, out):
        """The adapter merged into the original bf16 weights, on the CPU."""
        shutil.rmtree(out, ignore_errors=True)
        base = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cpu")
        merged = PeftModel.from_pretrained(base, str(adapter)).merge_and_unload()
        merged.save_pretrained(out)
        AutoProcessor.from_pretrained(MODEL).save_pretrained(out)
        tok.save_pretrained(out)
        del base, merged
        gc.collect()

    # what to export (file name, adapter): each curve point or the one adapter, and the earlier version this
    # one was updated from (its GGUF serves the tasks this one was not trained on)
    POINTS = ([(f"{ADAPTER}-{p.name}", p) for p in sorted((OUT / "curve").glob("read-*"))] if CURVE and not TRIAL else
              [(ADAPTER, OUT / "lora")])
    if R_DPO and (OUT / "lora-dpo").exists():
        tok.save_pretrained(str(OUT / "lora-dpo"))
        POINTS.append((f"{ADAPTER}-dpo", OUT / "lora-dpo"))
    if INIT_FROM and not TRIAL:
        POINTS.append((MANIFEST["parent"], INIT_LORA))
    print("export:", [n for n, _ in POINTS])


In [ ]:
if DONE or TRIAL:
    from huggingface_hub import snapshot_download

    (OUT / "gguf").mkdir(parents=True, exist_ok=True)
    # the server's build (b11368): a newer converter can write a GGUF the server cannot read
    !git clone --depth 1 --branch b11368 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
    !pip install -q /tmp/llama.cpp/gguf-py sentencepiece
    BASE_DIR = snapshot_download(MODEL)  # the base's config and tokenizer (already downloaded)
    for tag, adapter in POINTS:
        !python /tmp/llama.cpp/convert_lora_to_gguf.py {adapter} --base {BASE_DIR} --outtype f16 --outfile {OUT}/gguf/{tag}.gguf
    if MERGE and not TRIAL:
        !cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
        !cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 > /dev/null
        MERGED = Path("/tmp/merged")
        for tag, adapter in POINTS[:-1] if INIT_FROM else POINTS:  # not the parent (appended last)
            merge(adapter, MERGED)
            # --no-mtp: the converter counts Qwen3.5's MTP layer in block_count, but the merged
            # checkpoint has no mtp.* weights (transformers drops them), so blk.32 would be missing
            !python /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/merged --no-mtp --outtype bf16 --outfile /tmp/model-bf16.gguf
            !/tmp/llama.cpp/build/bin/llama-quantize /tmp/model-bf16.gguf {OUT}/gguf/{tag}-merged.Q8_0.gguf Q8_0
            !rm -f /tmp/model-bf16.gguf
    for f in sorted(Path("/kaggle/working").rglob("*.gguf")):
        print(f, f"{f.stat().st_size / 2**20:.0f} MB")


## Check the GGUFs

Each exported model must have weights for every layer its header counts, as `llama-server` checks on load. Qwen3.5's config counts an MTP layer the fine-tuned checkpoint does not have, and a converter that counts it writes a file the server rejects ("tensor 'blk.32.attn_norm.weight' not found"). Such a file is renamed `*.broken` and the run fails here instead of on the server.

In [ ]:
def check_gguf(path):
    """Every layer the GGUF's header counts has weights, and none beyond. A converter can count a layer the
    checkpoint does not have (Qwen3.5's MTP head, which transformers drops: the server then fails with
    "tensor 'blk.32.attn_norm.weight' not found"). Such a file is renamed *.broken so it is not downloaded."""
    from gguf import GGUFReader
    reader = GGUFReader(str(path))

    def field(key, default=None):
        f = reader.fields.get(key)
        return default if f is None else f.contents()

    arch = field("general.architecture")
    n, n_mtp = field(f"{arch}.block_count"), field(f"{arch}.nextn_predict_layers", 0)
    if n is None:  # a LoRA adapter has no layer count of its own
        print(f"{Path(path).name}: adapter, {len(reader.tensors)} tensors")
        return True
    layers = {int(t.name.split(".")[1]) for t in reader.tensors if t.name.startswith("blk.")}
    missing, extra = sorted(set(range(n)) - layers), sorted(layers - set(range(n)))
    del reader  # it maps the file, which cannot be renamed while mapped on Windows
    if not missing and not extra:
        print(f"{Path(path).name}: {n} layers, all with weights")
        return True
    broken = Path(str(path) + ".broken")
    Path(path).rename(broken)
    print(f"BROKEN {Path(path).name}: the header counts {n} layers ({n_mtp} of them MTP) but has no weights "
          f"for {missing} (and weights beyond the count for {extra}); renamed to {broken.name}. "
          "Convert with --no-mtp, with the llama.cpp the server runs.")
    return False


if DONE or TRIAL:
    ok = [check_gguf(f) for f in sorted((OUT / "gguf").glob("*.gguf"))]
    assert all(ok), "a GGUF the server cannot load (see BROKEN above)"


## Next: serve it

Download `gguf/<ADAPTER>.gguf` from the version's *Output* tab (keep `lora/`, `trained.json`
and `manifest.json` too: the next update starts from them). On the server, one base and every
adapter, none applied until a request asks for it:

```sh
llama-server -m Qwen3.5-4B-Base-Q8_0.gguf --lora nts-intents.gguf,nts-policy.gguf,nts-negotiation.gguf \
    --lora-init-without-apply --jinja -ngl 99 -c 8192 --port 8080
```

`curl <server>/lora-adapters` lists them. Name the adapter in `adapters.toml` and point the
project at it: `NTS_ADAPTERS=adapters.toml` for the web app, `--adapters adapters.toml` (or
`--adapter <name>` for one task) for the `--local` evaluations. Each request switches on its own
adapter (the `lora` field of the request). Thinking stays off, as in training. A server that cannot
take per-request adapters can serve each one as a model of its own name instead (a router preset of
the base with `lora = <file>`, or the `MERGE` GGUF): the client finds it by the same name.
